## 5.2 Simple RNN前向传播 - 维度变化与矩阵计算

#### 1. 这一节我们要解决什么问题 🎯

在上一小节中，我们已经从“流程”角度理解了 Simple RNN 的前向传播：

* 接收当前输入 $x_t$
* 接收上一时刻隐藏状态 $h_{t-1}$
* 计算当前隐藏状态 $h_t$
* 再由 $h_t$ 生成输出 $y_t$

但是，这些量到底是什么形状？

也就是说：

* $x_t$ 是一个数，还是一个向量？
* $h_t$ 是一个数，还是一组隐藏单元？
* $W_x$、$W_h$、$W_y$ 的维度应该怎么定？
* 如果一次输入一整个 batch，张量形状又该怎么变？
* $seq\_len$、$input\_size$、$hidden\_size$、$batch\_size$ 分别在什么位置上？

这一节的目标，就是把这些“维度问题”彻底讲清楚。

学完这一节后，你应该能够做到：

* 看懂 Simple RNN 每个变量的形状
* 看懂权重矩阵为什么要是那个维度
* 看懂单时间步输入和整段序列输入之间的关系
* 为后面学习 PyTorch 中的 `nn.RNN` 做准备


#### 2. 先记住 4 个最核心的概念 📦

##### 2.1 $input\_size$

表示：

> 每一个时间步输入特征的维度。

例如：

* 如果每个时间步只有 $1$ 个数值特征，那么 $input\_size = 1$
* 如果每个时间步有 $5$ 个特征，那么 $input\_size = 5$
* 如果一个单词被表示成 $300$ 维词向量，那么 $input\_size = 300$

所以它回答的问题是：

> “每一步输入的数据有多宽？”

##### 2.2 $hidden\_size$

表示：

> 隐藏状态向量的维度，也就是隐藏单元的个数。

例如：

* $hidden\_size = 8$，表示每个时间步有 $8$ 个隐藏单元
* 那么 $h_t$ 就是一个 $8$ 维向量

所以它回答的问题是：

> “RNN 内部要用多大的记忆向量来表示当前状态？”

##### 2.3 $seq\_len$

表示：

> 一个序列有多少个时间步。

例如一句话：

$I\ love\ deep\ learning$

如果按单词输入，那么：

* $I$
* $love$
* $deep$
* $learning$

一共 $4$ 个时间步，所以：

$seq\_len = 4$

所以它回答的问题是：

> “这条序列有多长？”

##### 2.4 $batch\_size$

表示：

> 一次同时送入模型的序列个数。

需要注意：

* batch 表示多个序列
* 而一个序列 $seq\_len$ 包含多个时间步
* 每个时间步中又包含 $input\_size$ 或经过计算后形成 $hidden\_size$

例如：

* 一次送 $16$ 句话进去训练，那么 $batch\_size = 16$
* 一次送 $32$ 条时间序列进去训练，那么 $batch\_size = 32$

所以它回答的问题是：

> “一次并行处理多少个样本？”

#### 3. 先统一我们的矩阵乘法视角 🧠

##### 3.1 为什么这一节要统一视角？

为了和之前学习的 MLP、PyTorch 写法保持一致，这一节我们统一采用：

> 行向量视角

来理解矩阵计算。

也就是说，单个样本我们写成一行：

* 输入写成 $(1,\ input\_size)$
* 隐藏状态写成 $(1,\ hidden\_size)$
* 输出写成 $(1,\ output\_size)$

##### 3.2 单时间步公式怎么写？

这样一来，RNN 单时间步公式就可以写成和 MLP 很接近的形式：

$h_t = f(x_t @ W_x^T + h_{t-1} @ W_h^T + b)$

$y_t = g(h_t @ W_y^T + b_y)$

你会发现，它和熟悉的全连接层公式：

$y = x @ W^T + b$

本质上是同一种逻辑。

##### 3.3 RNN 和 MLP 的差别在哪里？

RNN 比 MLP 多了一项：

$h_{t-1} @ W_h^T$

所以：

> RNN 单时间步中的线性变换，本质上就是两个线性层结果相加：一个来自当前输入，一个来自上一时刻隐藏状态。


#### 4. 单时间步下，各个变量的形状是什么 🧩

##### 4.1 我们先只看最简单情况

为了避免一开始就被 batch 和序列长度绕晕，我们先只看：

* 没有 batch
* 只看一个时间步

##### 4.2 输入 $x_t$ 的形状

在单时间步下，$x_t$ 表示当前时刻的输入特征向量。

这里我们按照熟悉的 MLP 写法，把单样本写成一行，因此：

$x_t.shape = (1,\ input\_size)$

例如：

如果当前时间步有 $3$ 个特征，那么：

$x_t.shape = (1,\ 3)$

这表示：

* 只有 $1$ 条样本
* 这一条样本有 $input\_size$ 个特征

这和之前 MLP 中单样本输入写成 $(1,\ n)$ 是同一种思路。

##### 4.3 上一时刻隐藏状态 $h_{t-1}$ 的形状

隐藏状态表示模型内部的记忆向量，所以它的长度由 $hidden\_size$ 决定。

同样地，我们也按“单样本一行”的方式来写：

$h_{t-1}.shape = (1,\ hidden\_size)$

例如：

如果 $hidden\_size = 5$，那么：

$h_{t-1}.shape = (1,\ 5)$

##### 4.4 当前隐藏状态 $h_t$ 的形状

因为当前隐藏状态和上一时刻隐藏状态本质上属于同一种量，所以它们维度相同。

因此：

$h_t.shape = (1,\ hidden\_size)$

##### 4.5 输出层 $y_t$ 的形状

输出层的维度由任务决定。

如果输出层维度记为 $output\_size$，那么：

$y_t.shape = (1,\ output\_size)$

例如：

* 二分类时，可能 $output\_size = 1$
* 三分类时，可能 $output\_size = 3$
* 回归一个值时，可能 $output\_size = 1$

所以：

> 当前只有 $1$ 条样本， output_size 表示输出结果有几个。

##### 4.6 这一部分的核心理解

在“单时间步、单样本”的情况下，我们统一写成二维形式：

* $x_t.shape = (1,\ input\_size)$
* $h_{t-1}.shape = (1,\ hidden\_size)$
* $h_t.shape = (1,\ hidden\_size)$
* $y_t.shape = (1,\ output\_size)$

这样写的好处是：

> 它和我们之前学习 MLP 时的矩阵乘法写法完全一致，更方便理解后面的线性变换。

#### 5. 单时间步下，权重矩阵的计算 🔧

这是理解张量维度最核心的一步。

我们继续使用统一公式：

$h_t = f(x_t @ W_x^T + h_{t-1} @ W_h^T + b)$

$y_t = g(h_t @ W_y^T + b_y)$

##### 5.1 $W_x$ 的形状

公式里有：

$x_t @ W_x^T$

我们知道：

* $x_t.shape = (1,\ input\_size)$
* 结果要映射到隐藏状态空间，也就是 $(1,\ hidden\_size)$

那么为了让矩阵乘法成立，$W_x$ 必须把：

> 输入空间 $\rightarrow$ 映射到 $\rightarrow$ 隐藏空间

所以：

$W_x.shape = (hidden\_size,\ input\_size)$

例如：

* $input\_size = 3$
* $hidden\_size = 5$

那么：

$W_x.shape = (5,\ 3)$

矩阵乘法：

$(1,\ 3) @ (3,\ 5) = (1,\ 5)$

所以最终结果就是：

$(1,\ hidden\_size)$

这正好就是隐藏状态空间的形状。  
类似于 MLP 中从输入层到下一个隐藏层，下一个隐藏层的 shape 为 $(1,\ hidden\_size)$。

##### 5.2 $W_h$ 的形状

公式里还有：

$h_{t-1} @ W_h^T$

我们知道：

* $h_{t-1}.shape = (1,\ hidden\_size)$
* 结果仍然要落在隐藏状态空间 $(1,\ hidden\_size)$

所以：

$W_h.shape = (hidden\_size,\ hidden\_size)$

例如：

如果 $hidden\_size = 5$，那么：

$W_h.shape = (5,\ 5)$

于是：

* $W_h^T.shape = (5,\ 5)$
* $h_{t-1}.shape = (1,\ 5)$

矩阵乘法：

$(1,\ 5) @ (5,\ 5) = (1,\ 5)$

所以结果仍然是：

$(1,\ hidden\_size)$

这表示：

> 上一时刻的隐藏状态，要经过一个“隐藏到隐藏”的变换，再参与当前状态计算。

##### 5.3 偏置 $b$ 的形状

因为：

$x_t @ W_x^T + h_{t-1} @ W_h^T$

的结果形状是：

$(1,\ hidden\_size)$

所以偏置 $b$ 必须也能加到这个向量上。

因此：

$b.shape = (1,\ hidden\_size)$

所以你可以这样理解：

* 数学上为了和当前样本对齐，可以看成 $(1,\ hidden\_size)$
* 代码实现中常直接写成 $(hidden\_size,)$，进行自动广播

##### 5.4 $W_y$ 的形状

输出公式是：

$y_t = g(h_t @ W_y^T + b_y)$

我们知道：

* $h_t.shape = (1,\ hidden\_size)$
* 输出希望得到 $(1,\ output\_size)$

所以：

$W_y.shape = (output\_size,\ hidden\_size)$

于是：

$W_y^T.shape = (hidden\_size,\ output\_size)$

矩阵乘法：

$(1,\ hidden\_size) @ (hidden\_size,\ output\_size) = (1,\ output\_size)$

也就是：

$y_t.shape = (1,\ output\_size)$

##### 5.5 $b_y$ 的形状

输出层偏置 $b_y$ 加在输出空间上，所以：

* 严格按二维理解，可以写成：  
  $b_y.shape = (1,\ output\_size)$
* 实际代码中更常简写成：  
  $b_y.shape = (output\_size,)$

实际上会自动广播。


#### 6. 把单时间步的所有形状统一起来 🧾

##### 6.1 先设定符号

假设：

* $input\_size = d_x$
* $hidden\_size = d_h$
* $output\_size = d_y$

##### 6.2 输入到隐藏状态

那么单时间步下：

* $x_t.shape = (1,\ d_x)$
* $W_x.shape = (d_h,\ d_x)$
* $h_{t-1}.shape = (1,\ d_h)$
* $W_h.shape = (d_h,\ d_h)$
* $b.shape = (1,\ d_h)$
* $h_t.shape = (1,\ d_h)$

##### 6.3 隐藏状态到输出

* $W_y.shape = (d_y,\ d_h)$
* $b_y.shape = (d_y,)$
* $y_t.shape = (1,\ d_y)$

#### 7. 从单时间步扩展到整个序列 ⏱️

##### 7.1 输入 $X$ 的形状

前面只是看一个时间步，但 RNN 真正处理的是整个序列。

假设一条序列长度为：

$seq\_len = T$

那么输入其实不是一个 $x_t$，而是：

* $x_1$
* $x_2$
* $x_3$
* $\dots$
* $x_T$

也就是说，一条序列本质上是：

> 由多个时间步输入向量组成的集合。

如果不考虑 batch，那么整条序列的输入可以写成：

$X.shape = (seq\_len,\ input\_size)$

也就是：

$X.shape = (T,\ d_x)$

你可以理解为：

* 一共有 $T$ 行
* 每一行是一个时间步的输入向量
* 每个向量长度是 $input\_size$

也就是：

> 把多个时间步的输入行向量，按时间顺序堆叠起来形成的二维张量。

##### 7.2 隐藏状态 $H$ 的形状

因为每个时间步都会生成一个隐藏状态：

* $h_1$
* $h_2$
* $h_3$
* $\dots$
* $h_T$

每个 $h_t$ 的形状都是 $(hidden\_size,)$。

所以，如果把整条序列的所有隐藏状态收集起来，那么形状就是：

$H.shape = (seq\_len,\ hidden\_size)$

也就是：

$H.shape = (T,\ d_h)$

这表示：

* 一共有 $T$ 个时间步
* 每个时间步对应一个 $d_h$ 维隐藏状态向量

##### 7.3 输出 $Y$ 的形状

这里要分两种情况。

* 情况一：每个时间步都输出
* 情况二：只在最后一步输出

##### 7.4 情况一：每个时间步都输出

例如在 $N:N$ 任务中，模型每读入一个时间步，就要产生一个对应输出：

* $y_1$
* $y_2$
* $y_3$
* $\dots$
* $y_T$

每个 $y_t$ 的形状都是 $(output\_size,)$。

所以整条序列所有输出堆起来后：

$Y.shape = (seq\_len,\ output\_size)$

也就是：

$Y.shape = (T,\ d_y)$

##### 7.5 情况二：只在最后一步输出

例如在 $N:1$ 任务中，模型会先读完整个序列，再在最后输出一个总结果。

这时虽然中间仍然会计算出：

* $h_1$
* $h_2$
* $h_3$
* $\dots$
* $h_T$

但最终输出时，通常只取：

* 最后一个隐藏状态 $h_T$
* 或者最后一个输出 $y_T$

因此，这时最终结果不是一整条序列输出 $Y$，而只是最后一步的结果：

* $y_T.shape = (1,\ output\_size)$
* 或者如果直接使用最后一个隐藏状态做分类：  
  $h_T.shape = (1,\ hidden\_size)$

#### 8. 整个序列时间下，权重矩阵的计算 🔧

##### 8.1 整个序列的前向传播，本质上仍然是单步递推

这一点非常重要。

很多人看到整条序列的输入张量以后，会以为：

> “是不是整个序列可以一次性像 MLP 那样整体做一个矩阵乘法？”

答案是：

> 不是。

##### 8.2 整个序列的核心，不是一次性整体矩阵乘法

前面在单时间步中，我们已经知道：

$h_t = f(x_t @ W_x^T + h_{t-1} @ W_h^T + b)$

$y_t = g(h_t @ W_y^T + b_y)$

这里如果按我们熟悉的 MLP / PyTorch 写法：

* $x_t.shape = (1,\ input\_size)$
* $h_{t-1}.shape = (1,\ hidden\_size)$

那么在单时间步下：

* $x_t @ W_x^T$ 的结果是 $(1,\ hidden\_size)$
* $h_{t-1} @ W_h^T$ 的结果也是 $(1,\ hidden\_size)$

然后两者相加，再经过激活函数，得到：

$h_t.shape = (1,\ hidden\_size)$

但是：

> RNN 真正的核心计算，是单时间步的重复计算。

##### 8.3 整个序列，本质上只是反复执行单时间步计算

假设一条序列长度为：

$seq\_len = T$

那么它包含：

* $x_1 \in (1,\ input\_size)$
* $x_2 \in (1,\ input\_size)$
* $x_3 \in (1,\ input\_size)$
* $\dots$
* $x_T \in (1,\ input\_size)$

RNN 在处理整条序列时，并不是先把整条序列一次性整体变成隐藏状态，而是：

> 分别对每个时间步，执行单个时间步的计算过程。

也就是：

* 第 $1$ 个时间步：  
  $h_1 = f(x_1 @ W_x^T + h_0 @ W_h^T + b)$
* 第 $2$ 个时间步：  
  $h_2 = f(x_2 @ W_x^T + h_1 @ W_h^T + b)$
* 第 $3$ 个时间步：  
  $h_3 = f(x_3 @ W_x^T + h_2 @ W_h^T + b)$
* 一直到第 $T$ 个时间步：  
  $h_T = f(x_T @ W_x^T + h_{T-1} @ W_h^T + b)$

所以：

> 整个序列不是一种新的计算方式，而只是把单时间步计算沿着时间维度重复了很多次。


#### 9. 再加入 batch 之后，形状怎么变 🚚

##### 9.1 为什么 batch 会让人混淆？

真实训练中，我们通常不是一次只送一条序列，而是一次送很多条。

这时就要加入 $batch\_size$。

在 RNN 里确实会出现两个“看起来都像一组数据”的维度：

* 一个是 $seq\_len$：同一条样本内部的多个时间步
* 一个是 $batch\_size$：一次并行送入模型的多条样本

关键就在于：它们不是同一种“多”。

* $seq\_len$ 的多：是同一条样本内部，按时间展开的多个 step，本质上是时间步的个数
* $batch\_size$ 的多：是多条不同样本并行计算，是个时间步中有多少输入在计算

##### 9.2 设定符号

假设：

* $batch\_size = B$
* $seq\_len = T$
* $input\_size = d_x$
* $hidden\_size = d_h$

##### 9.3 输入 $X$ 的形状

那么一个 batch 的输入张量，通常可以写成：

$X.shape = (B,\ T,\ d_x)$

这表示：

* 一共有 $B$ 条序列
* 每条序列长度是 $T$
* 每个时间步输入维度是 $d_x$

不过这里要注意：

> 不同框架的默认维度顺序可能不同。

例如 PyTorch 中：

* 默认 `nn.RNN` 输入形状是 $(T,\ B,\ d_x)$
* 如果设置 `batch_first=True`，则是 $(B,\ T,\ d_x)$

所以：

> batch 本质上就是在序列外面再多包一层维度。

##### 9.4 隐藏状态 $H$ 的形状

这里也分成两个层次来看。

* 某一个时间步的隐藏状态
* 整个序列所有时间步的隐藏状态

##### 9.5 某一个时间步的隐藏状态

对于某一个时间步 $t$，如果是 batch 输入，那么这一时刻不再只有一个 $h_t$，而是：

* 第 $1$ 条样本一个隐藏状态
* 第 $2$ 条样本一个隐藏状态
* $\dots$
* 第 $B$ 条样本一个隐藏状态

所以某一时刻的隐藏状态形状可以理解为：

$(B,\ d_h)$

也就是：

> 一批样本，每条样本一个 $hidden\_size$ 维向量。

##### 9.6 整个序列所有时间步的隐藏状态

如果把整个 batch、整个序列的隐藏状态都保留下来，那么形状是：

* 若 `batch_first=True`：$(B,\ T,\ d_h)$
* 若默认时间优先：$(T,\ B,\ d_h)$

##### 9.7 输出 $Y$ 的形状

如果每个时间步都输出，那么输出张量形状和隐藏状态类似：

* 若 `batch_first=True`：$(B,\ T,\ d_y)$
* 若默认时间优先：$(T,\ B,\ d_y)$


#### 10. 加入 Batch 之后，权重矩阵的计算 🔧

##### 10.1 仍然从熟悉的单时间步公式出发

现在熟悉的单时间步公式是：

$h_t = f(x_t @ W_x^T + h_{t-1} @ W_h^T + b)$

我们之前在“单样本、单时间步”下写成：

* $x_t.shape = (1,\ input\_size)$
* $h_{t-1}.shape = (1,\ hidden\_size)$

##### 10.2 现在加上 batch 之后

对于某一个固定时间步 $t$，不再只有 $1$ 条样本，而是有 $B$ 条样本同时处在这个时间步。

所以此时：

* $x_t.shape = (B,\ input\_size)$
* $h_{t-1}.shape = (B,\ hidden\_size)$

注意，这里虽然整个输入张量是三维的：

$X.shape = (B,\ T,\ input\_size)$

但在真正进行第 $t$ 步计算时，我们会取出第 $t$ 个时间步对应的切片：

$X[:,\ t,\ :]$

它的形状就是：

$(B,\ input\_size)$

也就是：

> $B$ 条样本，每条样本在当前时间步的输入向量。

##### 10.3 当前时间步的矩阵乘法

假设：

* $x_t.shape = (B,\ input\_size)$
* $W_x.shape = (hidden\_size,\ input\_size)$

那么：

$x_t @ W_x^T$

形状是：

$(B,\ input\_size) @ (input\_size,\ hidden\_size) = (B,\ hidden\_size)$

这表示：

> 对 batch 中每一条样本，都把当前输入映射到隐藏空间。

同理：

如果：

* $h_{t-1}.shape = (B,\ hidden\_size)$
* $W_h.shape = (hidden\_size,\ hidden\_size)$

那么：

$h_{t-1} @ W_h^T$

形状是：

$(B,\ hidden\_size) @ (hidden\_size,\ hidden\_size) = (B,\ hidden\_size)$

所以二者相加：

$x_t @ W_x^T + h_{t-1} @ W_h^T + b$

结果仍然是：

$(B,\ hidden\_size)$

再经过激活函数：

$h_t.shape = (B,\ hidden\_size)$

##### 10.4 batch 计算下，参数矩阵形状并不会改变

虽然输入、隐藏状态和输出的形状变化了，但是参数矩阵的形状并不会因为时间步数量变化而变化。

如果定义：

* $input\_size = d_x$
* $hidden\_size = d_h$
* $output\_size = d_y$

那么无论 batch 是多少，始终有：

* $W_x.shape = (d_h,\ d_x)$
* $W_h.shape = (d_h,\ d_h)$
* $b.shape = (d_h,)$  
  或者严格按二维理解写成 $(1,\ d_h)$
* $W_y.shape = (d_y,\ d_h)$
* $b_y.shape = (d_y,)$  
  或者严格按二维理解写成 $(1,\ d_y)$

也就是说：

* $W_x$ 只关心：输入特征维度 $input\_size$ 如何映射到隐藏状态维度 $hidden\_size$
* $W_h$ 只关心：上一时刻隐藏状态如何映射到当前隐藏状态
* $W_y$ 只关心：隐藏状态如何映射到输出空间

这也是参数共享的核心。